# FMCG Global Demand Planning and Forecasting

## Notebook 07 – Feature Engineering

### Objective

This notebook transforms the cleaned dataset into a machine learning-ready dataset by creating predictive features while avoiding data leakage.

The engineered dataset will be used for forecasting daily FMCG demand.

In [38]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [39]:
df = pd.read_csv(
    "data/processed/fmcg_sales_clean.csv",
    parse_dates=["date"]
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [40]:
df = df.sort_values(
    ["sku_id", "store_id", "date"]
).reset_index(drop=True)

In [41]:
df["year"] = df["date"].dt.year
df["quarter"] = df["date"].dt.quarter
df["month"] = df["date"].dt.month
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_year"] = df["date"].dt.dayofyear

In [42]:
df["is_month_start"] = (
    df["date"].dt.is_month_start.astype(int)
)

df["is_month_end"] = (
    df["date"].dt.is_month_end.astype(int)
)

In [ ]:
group = df.groupby(["store_id", "sku_id"])

In [44]:
df["lag_1"] = group["units_sold"].shift(1)

In [45]:
df["lag_7"] = group["units_sold"].shift(7)

In [46]:
df["lag_30"] = group["units_sold"].shift(30)

In [47]:
df["lag_90"] = group["units_sold"].shift(90)

In [48]:
df["rolling_mean_7"] = (
    group["units_sold"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

In [49]:
df["rolling_mean_30"] = (
    group["units_sold"]
    .transform(lambda x: x.shift(1).rolling(30).mean())
)

In [50]:
df["rolling_std_7"] = (
    group["units_sold"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

In [51]:
df["discount_amount"] = (
    df["list_price"] *
    df["discount_pct"] / 100
)

df["effective_price"] = (
    df["list_price"] -
    df["discount_amount"]
)

In [52]:
df["inventory_cover"] = (
    df["stock_on_hand"] /
    (df["units_sold"] + 1)
)

In [53]:
df["low_stock"] = (
    df["stock_on_hand"] < 20
).astype(int)

In [54]:
df["promotion"] = df["promo_flag"]

In [55]:
df["promo_lag"] = (
    group["promo_flag"].shift(1)
)

In [56]:
df["heavy_rain"] = (
    df["rain_mm"] > 20
).astype(int)

In [57]:
df["hot_day"] = (
    df["temperature"] > 30
).astype(int)

In [58]:
encoder = LabelEncoder()

In [59]:
df["country_enc"] = encoder.fit_transform(df["country"])

In [60]:
df["city_enc"] = encoder.fit_transform(df["city"])

In [61]:
df["channel_enc"] = encoder.fit_transform(df["channel"])

In [62]:
df["brand_enc"] = encoder.fit_transform(df["brand"])

In [63]:
df["category_enc"] = encoder.fit_transform(df["category"])

In [64]:
df["sku_enc"] = encoder.fit_transform(df["sku_id"])

In [65]:
df = df.dropna().reset_index(drop=True)

In [66]:
features = [

"lag_1",
"lag_7",
"lag_30",
"lag_90",

"rolling_mean_7",
"rolling_mean_30",
"rolling_std_7",

"effective_price",
"discount_pct",
"promotion",

"stock_on_hand",
"inventory_cover",
"lead_time_days",

"temperature",
"rain_mm",

"is_weekend",
"is_holiday",
"month",
"week",
"quarter",

"country_enc",
"city_enc",
"channel_enc",
"category_enc",
"brand_enc",
"sku_enc"

]

In [67]:
target = "units_sold"

In [74]:
split_index = int(len(df) * 0.8)

train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

In [75]:
X_train = train[features]
X_test = test[features]

y_train = train[target]
y_test = test[target]

In [70]:
print("Training Features :", X_train.shape)
print("Training Target   :", y_train.shape)

print()

print("Testing Features  :", X_test.shape)
print("Testing Target    :", y_test.shape)

Training Features : (1009550, 26)
Training Target   : (1009550,)

Testing Features  : (0, 26)
Testing Target    : (0,)


In [71]:
import os

os.makedirs("data/processed", exist_ok=True)

df.to_csv(
    "data/processed/fmcg_feature_engineered.csv",
    index=False
)

print("Feature-engineered dataset saved successfully.")

Feature-engineered dataset saved successfully.


In [76]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(807640, 26)
(201910, 26)
(807640,)
(201910,)


In [77]:
X_train.to_csv(
    "data/processed/X_train.csv",
    index=False
)

X_test.to_csv(
    "data/processed/X_test.csv",
    index=False
)

y_train.to_csv(
    "data/processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "data/processed/y_test.csv",
    index=False)

print("Train/Test files updated successfully.")

Train/Test files updated successfully.


In [78]:
print(df["date"].min())
print(df["date"].max())

2021-04-01 00:00:00
2023-12-31 00:00:00
